In [0]:
from pytickersymbols import PyTickerSymbols
import yfinance as yf
from pyspark.sql.types import StructType, StructField, DateType, DoubleType, LongType, StringType, BooleanType
import pandas as pd

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS yfinance_pipeline_dev;
USE CATALOG yfinance_pipeline_dev;
CREATE SCHEMA IF NOT EXISTS stocks_dataset;
USE SCHEMA stocks_dataset;

In [0]:
stock_data = PyTickerSymbols()

dowjones_stocks = stock_data.get_stocks_by_index('DOW JONES')
dowjones_symbols = [stock['symbol'] for stock in dowjones_stocks]

In [0]:
collected_symbols = []

# for symbol_list in [nasdaq100_symbols, dowjones_symbols, sp500_symbols, sp600_symbols]:
for symbol in dowjones_symbols:
    if symbol not in collected_symbols and "." not in symbol:
        collected_symbols.append(symbol)

print(f"Total unique symbols: {len(collected_symbols)}")
print(collected_symbols[:20])  # Show first 20 for preview

In [0]:
def load_stocks_info(
    collected_symbols,
    table_name,
    batch_size=100
):
    stock_info_schema = StructType([
        StructField("symbol", StringType(), True),
        StructField("displayName", StringType(), True),
        StructField("country", StringType(), True),
        StructField("shortName", StringType(), True),
        StructField("longName", StringType(), True),
        StructField("inDowJones", BooleanType(), True),
        # StructField("inNasDaq100", BooleanType(), True),
        # StructField("inS&P500", BooleanType(), True),
        # StructField("inS&P600", BooleanType(), True),
        StructField("industry", StringType(), True),
        StructField("sector", StringType(), True),
        StructField("marketCap", LongType(), True)
    ])

    for i in range(0, len(collected_symbols), batch_size):
        current_batch = collected_symbols[i:i + batch_size]
        print(f"Processing batch {i//batch_size + 1}: {current_batch[0]} to {current_batch[-1]}...")

        tickers_obj = yf.Tickers(current_batch)
        info_rows = []
        for symbol in current_batch:
            try:
                info = tickers_obj.tickers[symbol].info
                row = {
                    "symbol": symbol,
                    "displayName": info.get("displayName"),
                    "country": info.get("country"),
                    "shortName": info.get("shortName"),
                    "longName": info.get("longName"),
                    "inDowJones": True if symbol in dowjones_symbols else False,
                    # "inNasDaq100": True if symbol in nasdaq100_symbols else False,
                    # "inS&P500": True if symbol in sp500_symbols else False,
                    # "inS&P600": True if symbol in sp600_symbols else False,
                    "industry": info.get("industry"),
                    "sector": info.get("sector"),
                    "marketCap": info.get("marketCap")
                }
                info_rows.append(row)
            except Exception as e:
                print(f"Error fetching info for {symbol}: {e}")

        info_df = pd.DataFrame(info_rows)
        info_spark_df = spark.createDataFrame(info_df, schema=stock_info_schema)
        info_spark_df.write.mode("append").saveAsTable("stocks_info")

    print("dow jones stocks processed!")

In [0]:
# spark.sql("DROP TABLE IF EXISTS stocks_info")

if not spark.catalog.tableExists("stocks_info"):
    load_stocks_info(collected_symbols=collected_symbols, table_name="stocks_info")

# stocks_info = spark.sql("SELECT * FROM stocks_info")
# stocks_info.createOrReplaceTempView("stocks_info")